In [1]:
import sys
import numpy as np
import tensorflow as tf
sys.path.append('..')
from src.LSTM import LSTM, binary_cross_entropy

import os
import json
import sys
from itertools import product
import numpy as np
import tensorflow as tf
sys.path.append('..')
from src.LSTM import LSTM
from src.metrics import evaluate_predictions

I0000 00:00:1795188136.967985  154553 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795188136.998971  154553 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795188137.956767  154553 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)


I0000 00:00:1795188139.513258  154553 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4191 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [2]:
def pick_device():
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        print("No GPU visible to TensorFlow, using CPU")
        return '/CPU:0'
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        with tf.device('/GPU:0'):
            tf.reduce_sum(tf.random.uniform((8, 8))).numpy()
        print(f"Training on GPU: {gpus[0].name}")
        return '/GPU:0'
    except (RuntimeError, ValueError, tf.errors.OpError) as e:
        print(f"GPU unusable, falling back to CPU: {e}")
        return '/CPU:0'

DEVICE = pick_device()

GPU unusable, falling back to CPU: Physical devices cannot be modified after being initialized


In [3]:
SEQ_LEN = 10
RBM_HIDDEN = 256
OUTPUT_DIM = 1001
BATCH = 8

In [4]:
net = LSTM(
    sequence_length=SEQ_LEN,
    input_dim=RBM_HIDDEN,
    output_dim=OUTPUT_DIM,
    hidden_units=128,
    dropout=0.2,
    learning_rate=1e-3
)
net.model.summary()

Model: "next_basket_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)     │ (None, 10, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ basket_output (Dense)           │ (None, 1001)           │       129,129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 326,249 (1.24 MB)

 Trainable params: 326,249 (1.24 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
assert net.model.input_shape == (None, SEQ_LEN, RBM_HIDDEN)
assert net.model.output_shape == (None, OUTPUT_DIM)
assert not any(isinstance(l, tf.keras.layers.Bidirectional) for l in net.model.layers)
print("Input and output shapes match the contract; LSTM is unidirectional")

Input and output shapes match the contract; LSTM is unidirectional


In [6]:
X_dummy = np.random.rand(BATCH, SEQ_LEN, RBM_HIDDEN).astype(np.float32)
y_dummy = (np.random.rand(BATCH, OUTPUT_DIM) < 0.01).astype(np.float32)

pred = net.model.predict(X_dummy, verbose=0)
assert pred.shape == (BATCH, OUTPUT_DIM)
assert pred.min() >= 0.0 and pred.max() <= 1.0
print(f"Prediction shape: {pred.shape}, min: {pred.min():.4f}, max: {pred.max():.4f}")

Prediction shape: (8, 1001), min: 0.3942, max: 0.6184


I0000 00:00:1795188140.414857  154678 cuda_dnn.cc:442] Loaded cuDNN version 92700


In [7]:
keras_loss = tf.keras.losses.BinaryCrossentropy()(y_dummy, pred).numpy()
draft_loss = binary_cross_entropy(tf.constant(y_dummy), tf.constant(pred)).numpy()
print(f"Keras BCE: {keras_loss:.6f}, draft BCE: {draft_loss:.6f}")
assert np.isclose(keras_loss, draft_loss, atol=1e-5)

Keras BCE: 0.697677, draft BCE: 0.697677


In [8]:
loss_before = net.model.evaluate(X_dummy, y_dummy, verbose=0)
net.model.fit(X_dummy, y_dummy, epochs=5, batch_size=BATCH, verbose=0)
loss_after = net.model.evaluate(X_dummy, y_dummy, verbose=0)
print(f"Loss before: {loss_before:.6f}, after 5 epochs on the dummy batch: {loss_after:.6f}")
assert loss_after < loss_before

Loss before: 0.697677, after 5 epochs on the dummy batch: 0.581052


In [9]:
PROC_DIR = '../data/processed'
REPORT_DIR = '../data/reports'
PRED_DIR = '../data/predictions'
MODEL_DIR = '../data/models'
VOCAB_SIZE = 1001
MAX_SEQ_LEN = 10
BATCH_SIZE = 256
MAX_EPOCHS = 30
PATIENCE = 3
SEED = 42
K_VALUES = [1, 3, 5, 9]
GRID = {
    'learning_rate': [1e-3, 3e-4],
    'dropout': [0.2, 0.4],
    'hidden_units': [64, 128],
}
for d in [REPORT_DIR, PRED_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [10]:
X_padded = np.load(f'{PROC_DIR}/X_padded.npy').astype(np.float32)
Y = np.load(f'{PROC_DIR}/y_target.npy').astype(np.float32)
X_dense = np.load(f'{PROC_DIR}/X_rbm_dense.npy').astype(np.float32)
splits = np.load(f'{PROC_DIR}/splits.npz')
train_ex = splits['train']
val_ex = splits['validation']
test_ex = splits['test']

assert X_dense.shape[:2] == X_padded.shape[:2]
HIDDEN = X_dense.shape[2]
print(f"X_padded: {X_padded.shape}, X_dense: {X_dense.shape}, Y: {Y.shape}")
print(f"Examples - train: {len(train_ex)}, validation: {len(val_ex)}, test: {len(test_ex)}")

X_padded: (14074, 10, 1001), X_dense: (14074, 10, 500), Y: (14074, 1001)
Examples - train: 38981, validation: 14074, test: 14074


In [11]:
real_mask = (X_padded.sum(axis=2, keepdims=True) > 0).astype(np.float32)
H_hist = X_dense * real_mask
S_multi = np.concatenate([X_padded, Y[:, None, :]], axis=1)
print(f"Masked dense history: {H_hist.shape}, labels: {S_multi.shape}")

Masked dense history: (14074, 10, 500), labels: (14074, 11, 1001)


In [12]:
def dense_window(u, p):
    L = min(p, MAX_SEQ_LEN)
    x = np.zeros((MAX_SEQ_LEN, HIDDEN), dtype=np.float32)
    x[MAX_SEQ_LEN - L:] = H_hist[u, p - L:p]
    return x


def materialize(examples):
    X = np.stack([dense_window(u, p) for u, p in examples]).astype(np.float32)
    y = S_multi[examples[:, 0], examples[:, 1]].astype(np.float32)
    return X, y


Xtr, ytr = materialize(train_ex)
Xva, yva = materialize(val_ex)
Xte, yte = materialize(test_ex)
print(f"Train X: {Xtr.shape}, y: {ytr.shape}")
print(f"Validation X: {Xva.shape}, y: {yva.shape}")
print(f"Test X: {Xte.shape}, y: {yte.shape}")

Train X: (38981, 10, 500), y: (38981, 1001)
Validation X: (14074, 10, 500), y: (14074, 1001)
Test X: (14074, 10, 500), y: (14074, 1001)


In [13]:
ITEM_DIM = VOCAB_SIZE - 1
ytr_items = ytr[:, 1:]
yva_items = yva[:, 1:]


def to_full(scores):
    return np.concatenate([np.zeros((len(scores), 1), np.float32), scores], axis=1)

In [14]:
def make_ds(X, y, shuffle):
    def gen():
        idx = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            b = idx[i:i + BATCH_SIZE]
            yield X[b], y[b]
    spec = (
        tf.TensorSpec((None, MAX_SEQ_LEN, HIDDEN), tf.float32),
        tf.TensorSpec((None, ITEM_DIM), tf.float32),
    )
    return tf.data.Dataset.from_generator(gen, output_signature=spec).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(Xtr, ytr_items, shuffle=True)
val_ds = make_ds(Xva, yva_items, shuffle=False)

In [15]:
p = np.clip(ytr_items.mean(axis=0), 1e-6, 1 - 1e-6)
PRIOR_LOGIT = np.log(p / (1 - p)).astype(np.float32)
print(f"Output bias init: min={PRIOR_LOGIT.min():.2f}, max={PRIOR_LOGIT.max():.2f}, size={PRIOR_LOGIT.shape}")

Output bias init: min=-7.74, max=-2.90, size=(1000,)


In [16]:
class EpochLog(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f"    epoch {epoch + 1:>2} | loss={logs.get('loss', float('nan')):.5f} | "
              f"val_loss={logs.get('val_loss', float('nan')):.5f}", flush=True)


class ValHitRatio(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = -1.0
        self.best_weights = None
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        scores = to_full(self.model.predict(Xva, batch_size=BATCH_SIZE, verbose=0))
        res = evaluate_predictions(scores, yva, K_VALUES)
        mhr = float(np.mean([res[k]['HitRatio'] for k in K_VALUES]))
        print(f"    epoch {epoch + 1:>2} | val mean HR={mhr:.4f}", flush=True)
        if mhr > self.best:
            self.best = mhr
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= PATIENCE:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


def train_config(cfg, device):
    with tf.device(device):
        net = LSTM(
            sequence_length=MAX_SEQ_LEN,
            input_dim=HIDDEN,
            output_dim=ITEM_DIM,
            hidden_units=cfg['hidden_units'],
            dropout=cfg['dropout'],
            learning_rate=cfg['learning_rate'],
        )
        net.model.layers[-1].bias.assign(PRIOR_LOGIT)
        callbacks = [
            EpochLog(),
            ValHitRatio(),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5),
        ]
        hist = net.model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=MAX_EPOCHS,
            callbacks=callbacks,
            verbose=0,
        )
    return net, hist


def score_split(net, X):
    with tf.device(DEVICE):
        raw = net.model.predict(X, batch_size=BATCH_SIZE, verbose=0).astype(np.float32)
    return to_full(raw)

In [17]:
grid_results = []
nets = []
for lr, do, hu in product(GRID['learning_rate'], GRID['dropout'], GRID['hidden_units']):
    cfg = {'learning_rate': lr, 'dropout': do, 'hidden_units': hu}
    net, hist = train_config(cfg, DEVICE)
    val_scores = score_split(net, Xva)
    res = evaluate_predictions(val_scores, yva, K_VALUES)
    mean_hr = float(np.mean([res[k]['HitRatio'] for k in K_VALUES]))
    entry = {
        'config': cfg,
        'best_val_loss': float(min(hist.history['val_loss'])),
        'epochs_run': len(hist.history['loss']),
        'val_hit_ratio': {str(k): res[k]['HitRatio'] for k in K_VALUES},
        'val_mean_hit_ratio': mean_hr,
    }
    grid_results.append(entry)
    nets.append(net)
    print(f"lr={lr} dropout={do} hidden={hu} | val_loss={entry['best_val_loss']:.5f} "
          f"| val mean HR={mean_hr:.4f} | epochs={entry['epochs_run']}")

/home/parth/Projects/BITS/DM/next-basket-prediction/.venv/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1795188146.751904  154553 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


    epoch  1 | loss=0.01897 | val_loss=0.02043
    epoch  1 | val mean HR=0.1642
    epoch  2 | loss=0.01890 | val_loss=0.02043
    epoch  2 | val mean HR=0.1588
    epoch  3 | loss=0.01886 | val_loss=0.02040
    epoch  3 | val mean HR=0.1600
    epoch  4 | loss=0.01884 | val_loss=0.02040
    epoch  4 | val mean HR=0.1595
lr=0.001 dropout=0.2 hidden=64 | val_loss=0.02040 | val mean HR=0.1642 | epochs=4
    epoch  1 | loss=0.01897 | val_loss=0.02042
    epoch  1 | val mean HR=0.1627
    epoch  2 | loss=0.01889 | val_loss=0.02042
    epoch  2 | val mean HR=0.1606
    epoch  3 | loss=0.01885 | val_loss=0.02037
    epoch  3 | val mean HR=0.1627
    epoch  4 | loss=0.01882 | val_loss=0.02034
    epoch  4 | val mean HR=0.1637
    epoch  5 | loss=0.01879 | val_loss=0.02033
    epoch  5 | val mean HR=0.1642
    epoch  6 | loss=0.01878 | val_loss=0.02032
    epoch  6 | val mean HR=0.1643
    epoch  7 | loss=0.01877 | val_loss=0.02032
    epoch  7 | val mean HR=0.1643
    epoch  8 | loss=0.01876

In [18]:
best_idx = int(np.argmax([r['val_mean_hit_ratio'] for r in grid_results]))
best_cfg = grid_results[best_idx]['config']
best_net = nets[best_idx]
print(f"Selected on validation: {best_cfg}")
best_net.model.save(f'{MODEL_DIR}/lstm_best.keras')

Selected on validation: {'learning_rate': 0.0003, 'dropout': 0.4, 'hidden_units': 64}


In [19]:
test_scores = score_split(best_net, Xte)
test_res = evaluate_predictions(test_scores, yte, K_VALUES)

with open(f'{REPORT_DIR}/phase3_baseline_report.json') as f:
    base = json.load(f)

print(f"{'K':>4} | {'Model':<13} | {'HitRatio':>9} | {'Recall':>9}")
for k in K_VALUES:
    print(f"{k:>4} | {'LSTM':<13} | {test_res[k]['HitRatio']:>9.4f} | {test_res[k]['Recall']:>9.4f}")
    for label, key in [('Most Popular', 'most_popular'), ('FP-Growth', 'fpgrowth')]:
        r = base[key][str(k)]
        print(f"{k:>4} | {label:<13} | {r['HitRatio']:>9.4f} | {r['Recall']:>9.4f}")

for k in K_VALUES:
    for label, key in [('Most Popular', 'most_popular'), ('FP-Growth', 'fpgrowth')]:
        base_hr = base[key][str(k)]['HitRatio']
        lift = (test_res[k]['HitRatio'] - base_hr) / max(base_hr, 1e-12)
        print(f"K={k}: LSTM HitRatio lift over {label}: {lift:+.2%}")

   K | Model         |  HitRatio |    Recall
   1 | LSTM          |    0.1368 |    0.0601
   1 | Most Popular  |    0.1533 |    0.0707
   1 | FP-Growth     |    0.1533 |    0.0707
   3 | LSTM          |    0.1998 |    0.0927
   3 | Most Popular  |    0.1998 |    0.0927
   3 | FP-Growth     |    0.2034 |    0.0930
   5 | LSTM          |    0.2267 |    0.1032
   5 | Most Popular  |    0.2347 |    0.1044
   5 | FP-Growth     |    0.2374 |    0.1051
   9 | LSTM          |    0.3054 |    0.1389
   9 | Most Popular  |    0.2796 |    0.1219
   9 | FP-Growth     |    0.2829 |    0.1234
K=1: LSTM HitRatio lift over Most Popular: -10.71%
K=1: LSTM HitRatio lift over FP-Growth: -10.71%
K=3: LSTM HitRatio lift over Most Popular: +0.00%
K=3: LSTM HitRatio lift over FP-Growth: -1.78%
K=5: LSTM HitRatio lift over Most Popular: -3.41%
K=5: LSTM HitRatio lift over FP-Growth: -4.52%
K=9: LSTM HitRatio lift over Most Popular: +9.20%
K=9: LSTM HitRatio lift over FP-Growth: +7.93%


In [20]:
np.save(f'{PRED_DIR}/lstm_test_scores.npy', test_scores)
report = {
    'grid_results': grid_results,
    'selected_config': best_cfg,
    'test_lstm': {str(k): v for k, v in test_res.items()},
}
with open(f'{REPORT_DIR}/lstm_training_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f"Saved test scores to {PRED_DIR} and report to {REPORT_DIR}")

Saved test scores to ../data/predictions and report to ../data/reports


In [21]:
s = test_scores[:, 1:]
item_mean = s.mean(axis=0)
total_var = s.var()
item_part = item_mean.var()
user_part = (s - item_mean).var()
print(f"Variance explained by item identity (popularity): {item_part / total_var:.2%}")
print(f"Variance explained by user (personalization):     {user_part / total_var:.2%}")

Variance explained by item identity (popularity): 99.19%
Variance explained by user (personalization):     0.81%
